# Hadoop環境のアップグレード — 計画と準備

D12で構築した環境を更新する。ここでは更新前の状態を確認し、配布物の準備と利用停止を行う。

## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*

# 操作対象クラスタの設定

D10で作成したinventoryとgroup_varsを使用する。

In [ ]:
target_group = 'hadoop_all_testcluster'
target_group

In [ ]:
!ansible {target_group} -m ping

In [ ]:
import yaml
with open('group_vars/' + target_group) as configuration:
    kerberos_enabled = 'kerberos_realm' in yaml.safe_load(configuration)
kerberos_enabled

# 管理操作の認証

Kerberos構成の場合、HDFSの管理操作を行うホストでhdfsユーザーのkeytabを使って認証する。

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_namenode:hadoop_client' -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'

# Playbookの準備

In [ ]:
import os
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

使用するNotebookと同じ版のPlaybookを取得する。

In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1

In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir

# 更新前の状態

対象環境のバックアップと復元手順を確認する。利用停止の範囲と時間を関係者と合意してから進む。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -rollingUpgrade query' -l {target_group}

Active 1台・Standby 1台であり、進行中のrolling upgradeがないことを確認する。

## 更新前データの記録

更新中に変更しないHDFSファイルを選ぶ。O12jで照合する。

In [ ]:
verification_path = '/dataset/iris/iris.data'
verification_path

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs dfs -checksum {verification_path}" -l {target_group}

# 更新する配布物の指定

更新する版と配布物のURL・SHA-512を指定する。[HBaseとHadoopの対応](https://hbase.apache.org/docs/configuration/basic-prerequisites/)と、各版のリリースノートを確認する。

In [ ]:
hadoop_version = '3.3.6'
hadoop_package_filename = 'hadoop-3.3.6'
hadoop_release_url = 'https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz'
hadoop_release_checksum = 'sha512:de3eaca2e0517e4b569a88b63c89fae19cb8ac6c01ff990f1ff8f0cc0f3128c8e8a23db01577ca562a0e0bb1b4a3889f8c74384e609cd55e537aada3dcaa9f8a'
(hadoop_version, hadoop_package_filename, hadoop_release_url, hadoop_release_checksum)

In [ ]:
zookeeper_version = '3.6.4'
zookeeper_package_filename = 'apache-zookeeper-3.6.4-bin'
zookeeper_release_url = 'https://archive.apache.org/dist/zookeeper/zookeeper-3.6.4/apache-zookeeper-3.6.4-bin.tar.gz'
zookeeper_release_checksum = 'sha512:dec4835a1a23fa69599091d03b35f6c9f4e545d5fa1e149658cffdd7d1044ec31e14f41189aabbf59011dfce6609e6850209008e67cfd0f26270f1558196dcd8'
(zookeeper_version, zookeeper_package_filename, zookeeper_release_url, zookeeper_release_checksum)

In [ ]:
hbase_version = '2.5.10'
hbase_package_filename = 'hbase-2.5.10-hadoop3'
hbase_release_url = 'https://archive.apache.org/dist/hbase/2.5.10/hbase-2.5.10-hadoop3-bin.tar.gz'
hbase_release_checksum = 'sha512:fc663cd60d1ac0f8270b022532e7538d479079a862a03945e081ccf39821357635f3a7fdddbd1f10d45021b54b170ec774337dd4523414162c840204a4daced9'
(hbase_version, hbase_package_filename, hbase_release_url, hbase_release_checksum)

In [ ]:
spark_version = '3.3.4'
spark_package_filename = 'spark-3.3.4-bin-hadoop3'
spark_release_url = 'https://archive.apache.org/dist/spark/spark-3.3.4/spark-3.3.4-bin-hadoop3.tgz'
spark_release_checksum = 'sha512:a3874e340a113e95898edfa145518648700f799ffe2d1ce5dde7743e88fdf5559d79d9bcb1698fdfa5296a63c1d0fc4c8e32a93529ed58cd5dcf0721502a1fc7'
(spark_version, spark_package_filename, spark_release_url, spark_release_checksum)

In [ ]:
jsvc_version = '1.3.4'
jsvc_package_filename = 'commons-daemon-1.3.4-src'
jsvc_release_url = 'https://archive.apache.org/dist/commons/daemon/source/commons-daemon-1.3.4-src.tar.gz'
jsvc_release_checksum = 'sha512:bb36d88bc21a5777245012b2a73ee0e764b85715731f54cc4ff09343e95ccb18fc6c68b3ae9c680fb45a60c7ef5ed0f9e40991c2c03246dd7f8dd65031eddf24'
(jsvc_version, jsvc_package_filename, jsvc_release_url, jsvc_release_checksum)

# 更新先の設定

現在の対象クラスタの設定を控える。

In [ ]:
!cp group_vars/{target_group} {work_dir}/{target_group}-old

[group_vars](../tree/group_vars/)の対象クラスタのファイルを開き、以下の配布物の指定を更新する。

In [ ]:
print(f"""hadoop_package_filename: {hadoop_package_filename}
hadoop_release_url: {hadoop_release_url}
hadoop_release_checksum: {hadoop_release_checksum}
zookeeper_package_filename: {zookeeper_package_filename}
zookeeper_release_url: {zookeeper_release_url}
zookeeper_release_checksum: {zookeeper_release_checksum}
hbase_package_filename: {hbase_package_filename}
hbase_release_url: {hbase_release_url}
hbase_release_checksum: {hbase_release_checksum}
spark_package_filename: {spark_package_filename}
spark_release_url: {spark_release_url}
spark_release_checksum: {spark_release_checksum}
jsvc_package_filename: {jsvc_package_filename}
jsvc_release_url: {jsvc_release_url}
jsvc_release_checksum: {jsvc_release_checksum}""")

In [ ]:
! ! diff -u {work_dir}/{target_group}-old group_vars/{target_group}

変更が更新先の配布物に限られることを確認する。

# 配布物の配置

配布物を展開する。この段階ではcurrentを切り替えない。

In [ ]:
!ansible-playbook {playbook_dir}/stage_upgrade_packages.yml -l {target_group}

# 利用停止と実行中ジョブの確認

利用者のジョブ投入とHBaseクライアントを停止する。実行中のYARNアプリケーションが0件になるまで待つ。

In [ ]:
!ansible hadoop_client -a "yarn application -list -appStates SUBMITTED,ACCEPTED,RUNNING" -l {target_group}

# HBaseデータの記録

更新後の照合に使うテーブルを作る。既存のテーブルと重ならない名前を指定する。

In [ ]:
hbase_check_table = 'upgrade_check'
hbase_check_table

HBaseの管理操作はhbaseユーザーで行う。Kerberos構成では、サーバーの認証設定を使ってZooKeeperにも接続する。

In [ ]:
if kerberos_enabled:
    !ansible hadoop_hbase_master --become --become-user hbase -m shell -a 'kinit -kt /etc/hbase/conf/hbase.keytab hbase/$(hostname -f)' -l {target_group}

In [ ]:
hbase_query = f"""create '{hbase_check_table}', 'data'
put '{hbase_check_table}', 'before', 'data:value', 'preserved'
flush '{hbase_check_table}'
get '{hbase_check_table}', 'before'
"""
with open(os.path.join(work_dir, "hbase-before.rb"), "w") as query_file:
    query_file.write(hbase_query)
print(hbase_query)

In [ ]:
!ansible hadoop_hbase_master --become -m copy -a "src={work_dir}/hbase-before.rb dest=/tmp/hbase-upgrade-before.rb owner=hbase group=hbase mode=0644" -l {target_group}

In [ ]:
if kerberos_enabled:
    !ansible hadoop_hbase_master --become --become-user hbase -a "hbase --auth-as-server shell -n /tmp/hbase-upgrade-before.rb" -l {target_group}
else:
    !ansible hadoop_hbase_master --become --become-user hbase -a "hbase shell -n /tmp/hbase-upgrade-before.rb" -l {target_group}

# HBaseと履歴サービスの停止

利用者の処理が終了してから、O12bの停止順でサービスを停止する。

In [ ]:
!ansible-playbook {playbook_dir}/stop_spark_historyserver.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/stop_hbase_regionserver.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/stop_hbase_master.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/stop_mapreduce_historyserver.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/stop_timelineservice.yml -l {target_group}

[ZooKeeperの更新](O12f_Hadoop%20-%20Upgrade%20ZooKeeper.ipynb)へ進む。